In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.api.types import is_numeric_dtype
import plotly.express as px
import seaborn as sns
import math

# 1. Load and inspect the data

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/salioudunn11/Fitness_Prediction/main/gym_members_exercise_tracking.csv")

In [ ]:
display(df.head())

In [ ]:
display(df.shape)

In [ ]:
display(df.dtypes)

# 2. Summary Statistics

In [ ]:
display(df.describe())

In [ ]:
# it seems redundant but we wanted to see this four vlaues in clear format
display(df.describe().T[['mean', 'std', 'min', 'max']].round(2))

In [ ]:
display(df['Workout_Type'].value_counts())

In [ ]:
display(df['Gender'].value_counts())

# 3. Data Visualization

### 3.1 Calories Burned Distribution

In [ ]:
df['Calories_Burned'].hist(bins=10)
plt.title("Distribution of Calories Burned")
plt.xlabel("Calories")
plt.ylabel("Frequency")
plt.show()

### 3.2 Numerical Attribute Boxplots

In [ ]:
numeric_cols = [
    'Age', 'Weight (kg)', 'Height (m)', 'Max_BPM',
    'Avg_BPM', 'Resting_BPM', 'Session_Duration (hours)',
    'Calories_Burned', 'Fat_Percentage', 'Water_Intake (liters)',
    'Workout_Frequency (days/week)','Experience_Level', 'BMI'
]
# Keep only columns actually present in df
plot_cols = [c for c in numeric_cols if c in df.columns]

# 2. Setup grid dimensions
n_cols = 3
n_rows = math.ceil(len(plot_cols) / n_cols)

fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(14, n_rows * 3.2))
axes = axes.flatten()

# Custom style for outlier markers
flierprops = dict(marker='o', markersize=5, markerfacecolor='#e74c3c', markeredgecolor='none', alpha=0.7)

for i, col in enumerate(plot_cols):
    ax = axes[i]

    # width=0.35 keeps boxes sleek rather than filling the entire card
    sns.boxplot(
        y=df[col],
        ax=ax,
        width=0.35,
        color='#4a90e2',
        linewidth=1.2,
        flierprops=flierprops
    )

    # Clean styling
    ax.set_title(col, fontsize=11, fontweight='semibold', pad=10)
    ax.set_ylabel("")
    ax.grid(axis='y', linestyle=':', alpha=0.6)

# Remove unused axes in the grid
for j in range(len(plot_cols), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Outlier Analysis across Numerical Attributes", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.3 Correlation Matrix

In [ ]:
plt.figure(figsize=(12, 10))
numeric_cols = df.select_dtypes(include=[np.number])
correlation_matrix = numeric_cols.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Color-Coded Pearson Correlation Matrix')
plt.show()

### 3.4 Calories by Workout Type & Gender

In [ ]:
print("Correlation of each feature with 'Calories_Burned':")
display(correlation_matrix['Calories_Burned'].round(2).sort_values(ascending=False))

###3.5 Categorical vs. Target Analysis

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=df, x='Workout_Type', y='Calories_Burned', hue='Gender', errorbar=None, palette='muted')
plt.title("Mean Calories Burned Across Workout Types by Gender")
plt.ylabel("Average Calories Burned")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
fig = px.scatter_matrix(
    df,
    dimensions=['Session_Duration (hours)', 'Calories_Burned', 'Avg_BPM', 'Fat_Percentage'],
    color='Workout_Type',
    title="Scatter Matrix of Gym Attributes by Workout Type",
    width=900,
    height=900
)
fig.show()

# 4. Data Cleaning and Preprocessing

### 4.1 Missing Values & Duplicates

In [ ]:
display(df.isna().sum())

In [ ]:
display(df.duplicated().sum())

### 4.2 Unit Conversions

In [ ]:
# 1. Create an independent working copy to preserve raw data
df_clean = df.copy()

# 2. Add imperial conversions
df_clean['Weight (lb)'] = (df_clean['Weight (kg)'] * 2.20462).round(2)
df_clean['Height (in)'] = (df_clean['Height (m)'] * 39.3701).round(2)

# 3. Drop the metric columns
df_clean = df_clean.drop(columns=['Weight (kg)', 'Height (m)'])

# 4. Inspect result
display(df_clean.head())

### 4.3 Z-Score Outlier Detection

In [ ]:
num_data = df_clean.select_dtypes(include=[np.number])
Z = (num_data - num_data.mean()) / num_data.std()

def highlight_outliers(val):
    if abs(val) > 3:
        return 'background-color: #ff4d4d; color: white; font-weight: bold;'
    return 'color: #555555;'

# Filter rows that actually contain at least one outlier to keep it concise
outlier_rows = Z[(Z.abs() > 3).any(axis=1)]

styled_outliers = (
    outlier_rows.style
    .map(highlight_outliers)
    .format("{:+.2f}")
    .set_caption("🚩 Records with Extreme Outliers (|Z| > 3.0)")
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold')]},
        {'selector': 'caption', 'props': [('caption-side', 'top'), ('font-size', '14px'), ('font-weight', 'bold'), ('margin-bottom', '8px')]}
    ])
)

display(styled_outliers)

In [ ]:
outlier_mask = (Z.abs() > 3).any(axis=1)
outliers = df_clean.loc[outlier_mask]
print(f'Number of Total Outlier: {len(outliers)}')

# 5. Discretization

### 5.1 Equal-Width Binning

In [ ]:
cal_bins_width = pd.cut(df_clean['Calories_Burned'], 4)
print("Equal-width bins for Calories_Burned:")
display(cal_bins_width.value_counts(sort=False))

In [ ]:
fat_bins_width = pd.cut(df_clean['Fat_Percentage'], 4)
print("Equal-width bins for Fat_Percentage:")
display(fat_bins_width.value_counts(sort=False))

### 5.2 Equal-Frequency Binning

In [ ]:
cal_bins_freq = pd.qcut(df_clean['Calories_Burned'], 4)
print("Equal-frequency bins for Calories_Burned:")
display(cal_bins_freq.value_counts(sort=False))

In [ ]:
fat_bins_freq = pd.qcut(df_clean['Fat_Percentage'], 4)
print("Equal-frequency bins for Fat_Percentage:")
display(fat_bins_freq.value_counts(sort=False))